#英语时态学习

辅助学习英语时态。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI()

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def callGptMini(messages, model_override=None):
    # 型号 =“gpt-4o-mini”
    model = "gpt-5-mini"

    if model_override is not None:
        model = model_override

    response = openai.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content.strip()

## 句子生成指令

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def composeSystemSentenceGenerationInstructions(number_of_sentences=3):
    return f"""You are an English teacher who helps students improve their english.

Your task is to generate {number_of_sentences} sentence in user's language.
The sentences should be in the specified tense and follow the given theme.
The sentences should be of varying complexity, from simple to complex.
"""


def composeUserSentenceGenerationInstructions(language, tense, theme="any"):
    return f"""
  Language: {language}
  Tense: {tense}
  Theme: {theme}
"""


print(
    "Example:",
    composeUserSentenceGenerationInstructions("pl", "present perfect", "travel"),
)

定义一个简单的时态+主题组合

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 应该通过用户输入来获得
language = "pl"
tense = "past perfect"
theme = "travel"

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
sentence_generation_messages = [
    {"role": "system", "content": composeSystemSentenceGenerationInstructions(3)},
    {
        "role": "user",
        "content": composeUserSentenceGenerationInstructions(language, tense, theme),
    },
]

生成要翻译的句子。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
sentences_to_translate = callGptMini(sentence_generation_messages)
print(sentences_to_translate)
# 如果需要的话，我们可以将句子分成一个列表

让我们模拟用户并用一些超级便宜的模型来翻译它。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def composeUserTranslationInstructions(
    sentences, source_language, target_language, tense
):
    return f"""
  Translate the following sentences from {source_language} to {target_language}.
  The translations should use {tense} tense.

  Sentences:
  {sentences}
  """


translation_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that translates sentences accurately.",
    },
    {
        "role": "user",
        "content": composeUserTranslationInstructions(
            sentences_to_translate, language, "english", tense
        ),
    },
]

translations = callGptMini(translation_messages, model_override="gpt-5-nano")

print("Simulated translations:\n", translations)

创建提示来验证用户的翻译。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
def composeValidationSystemPrompt(user_language, tense, original_sentences):
    return f"""You are an English teacher who reviews students' sentences for grammar and vocabulary accuracy.
User's task was to translate sentences from {user_language} into English using {tense} tense.

Your task is to review the translations provided by the student.

# 指南
	- The sentences are specifically meant to be in {tense} tense.
  - For each sentence, provide feedback on grammar and vocabulary.
  - If the sentence is correct, respond with "Correct" and optionally provide some feedback.
  - If there are mistakes, provide the corrected sentence along with an explanation of the errors.
  - Keep your feedback concise and focused on the most important issues.
  - Verb tense usage is especially important; ensure the translations correctly use {tense} tense.

The original sentences were:
{original_sentences}

Now, analyze and review the user's translations.
"""


def composeValidationUserPrompt(user_translations):
    return f"""
  My translations:
  {user_translations}
  """


print(
    "Example:",
    composeValidationSystemPrompt("pl", tense, "sentence1\nsentence2\nsentence3"),
)
print(
    "Example:",
    composeValidationUserPrompt("My translation1\nMy translation2\nMy translation3"),
)

最后，让我们评估用户的努力。

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
validation_messages = [
    {
        "role": "system",
        "content": composeValidationSystemPrompt(
            language, tense, sentences_to_translate
        ),
    },
    {"role": "user", "content": composeValidationUserPrompt(translations)},
]

validation_feedback = callGptMini(validation_messages, model_override="gpt-5-mini")
print("🗒️ Validation feedback:\n", validation_feedback)

## 总结

工作流程运行良好。改进该项目的一件事是强制执行严格的格式。 

例如，生成的句子应从 1 到 X 编号。
每个点（句子）都应另起一行。
除了那些 X 句子之外，响应不应包含任何内容。

我们可以将用户正在学习的语言作为参数。

我们应该检查该模式知道的语言列表，并且不允许其他语言。